# Lab  – Reuse and Orchestrate AI Agents with Microsoft Foundry SDK (Python)

## Scenario
You will reuse the **Inventory**, **Insight** and **Action** agents created in Microsoft Foundry and interact with it from **Visual Studio Code** using **Python**. Then you will create a second agent and orchestrate both agents using the **Microsoft Agent Framework** workflow pattern.

> **Important:** In the new Foundry Agents architecture, agent behavior is defined at creation time (versions are static). Avoid trying to override tools/instructions at runtime.

## Prerequisites
- You have **Project Endpoint** + **ProductInventoryManager Agent ID**)
- Visual Studio Code + Microsoft Foundry extension installed
- Python 3.10+
- Azure CLI installed and authenticated: `az login`

### Values you need from Microsoft Foundry AI project
- `AZURE_FOUNDRY_PROJECT_ENDPOINT`
- `INVENTORY_AGENT_NAME`
- `INSIGHTS_AGENT_NAME`
- `ACTIONS_AGENT_NAME`

## Step 1 — (Optional) Connect VS Code to Foundry
1. Install the **Microsoft Foundry for VS Code** extension.
2. Sign in to Azure.
3. Set your **default Foundry project** so Agents/Models appear in the sidebar.

This step validates you are pointing at the same Foundry project as Lab 1.

## Step 2 — Install required packages
These packages are used in Microsoft Foundry quickstarts and SDK guidance for working with Foundry projects and agents.# Create your chat client and agents
deployment_name = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME", model_id)
chat_client = AzureOpenAIChatClient(
    credential=AzureCliCredential(),
    deployment_name=deployment_name,
)


In [ ]:
%pip install azure-ai-projects --pre
%pip install azure-identity openai python-dotenv

# Microsoft Agent Framework (Python)
%pip install agent-framework --pre

## Step 3 — Configure environment variables
Create a `.env` file in this folder with the following values.

In [ ]:
# Create/overwrite .env with your values
# (You can also set these in your shell instead.)

#AZURE_FOUNDRY_PROJECT_ENDPOINT="<paste-your-project-endpoint-here>"
#INVENTORY_AGENT_NAME="<paste-agent-id-here>"
#INSIGHTS_AGENT_NAME="<paste-agent-id-here>"
#ACTIONS_AGENT_NAME="<paste-agent-id-here>"
#AZURE_FOUNDRY_PROJECT_MODEL_ID="gpt-5-mini"

## Step 4 — Authenticate to Azure
Make sure you are logged in before running Python.

In [ ]:
# Run in terminal (not in notebook)
# if you haven't already authenticated with Azure CLI, do so now with:
# if you get error like comand az not found then run below in bash
# curl -sL https://aka.ms/InstallAzureCLIDeb | sudo bash
# az login

## Step 5 — Load config + connect to your Foundry project (Python)
This cell loads `.env` and creates a Foundry project client.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.agents import AgentsClient

load_dotenv()

project_endpoint = os.environ["AZURE_FOUNDRY_PROJECT_ENDPOINT"]
inventory_agent_name = os.environ["INVENTORY_AGENT_NAME"]
insights_agent_name = os.environ["INSIGHTS_AGENT_NAME"]
actions_agent_name = os.environ["ACTIONS_AGENT_NAME"]
model_id = os.getenv("AZURE_FOUNDRY_PROJECT_MODEL_ID", "gpt-5-mini")


credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
agent_client = AgentsClient( endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("Connected to project endpoint:", project_endpoint)
print("Reusing inventory agent name:", inventory_agent_name)
print("Reusing insights agent name:", insights_agent_name)
print("Reusing actions agent name:", actions_agent_name)
print("Default model id (for new agents):", model_id)


## Step 6 — Get the pointers to all agents by Name
This cell retrieves the persistent agent created in Mictosoft Foundry

In [ ]:
# The projects client exposes agent operations via .agents
# Retrieve existing agent
inventory_agent = project_client.agents.get(agent_name=inventory_agent_name)
print("Agent name:", getattr(inventory_agent, 'name', None))
insights_agent = project_client.agents.get(agent_name=insights_agent_name)
print("Agent name:", getattr(insights_agent, 'name', None))
actions_agent = project_client.agents.get(agent_name=actions_agent_name)
print("Agent name:", getattr(actions_agent, 'name', None))

## Step 7 — Ask the existing agent a question
This uses the OpenAI-compatible Responses API through the Foundry project.

> Note: Depending on SDK version, you may use `project_client.inference.get_openai_client()` or a similar helper. If your installed SDK exposes a different helper, use the code sample generated by Foundry/VS Code for your agent.

In [ ]:
# --- Minimal pattern using OpenAI-compatible client ---
# Handle SDK differences for obtaining the OpenAI-compatible client.
if hasattr(project_client, "inference") and hasattr(project_client.inference, "get_openai_client"):
    openai_client = project_client.inference.get_openai_client()
elif hasattr(project_client, "get_openai_client"):
    openai_client = project_client.get_openai_client()
else:
    candidates = [n for n in dir(project_client) if "openai" in n.lower() or "inference" in n.lower()]
    raise AttributeError(
        "Could not find an OpenAI client helper on project_client. "
        f"Found related attributes: {candidates}"
    )

#question = "Describe our top-selling product and give a short product description."
question = "<TODO: replace with your question for the agent>"
inventory_agent = getattr(inventory_agent, "name", inventory_agent_name)

response = openai_client.responses.create(
   input=question,
   extra_body={"agent": {"type": "agent_reference", "name": inventory_agent}},
)

print(response.output_text)

## Step 8 —  Build the sequential workflow with WorkflowBuilder

We create a directed pipeline:

`InventoryAgent` → `InsightsAgent` → `ActionsAgent`

The docs show the Python pattern `WorkflowBuilder(start_executor=...)` + `add_edge(...)` + `build()`.

### Agent	                        Responsibility

#### Inventory Agent ----               Detects low‑stock risks
#### Insights Agent ----------          Data Insight
#### Actions Agent	----------          Proposes targeted actions




In [ ]:
from typing import cast
from agent_framework import WorkflowBuilder, Agent, AgentResponse
from agent_framework.azure import AzureAIAgentClient


prompt = (
    'TODO: replace with your initial prompt for the agent(s) in the workflow'
)

inventory_agent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="InventoryAgent")
insight_agent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="InsightAgent")
action_agent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="ActionAgent")
coagent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="coagent")

coagent = coagent_client.as_agent(name="coagent")   
inventory_agent = inventory_agent_client.as_agent(name="InventoryAgent")
insight_agent = insight_agent_client.as_agent(name="InsightsAgent")
action_agent = action_agent_client.as_agent(name="ActionsAgent")

workflow = (
    WorkflowBuilder(start_executor=coagent)
    .add_edge(coagent, inventory_agent)
    .add_edge(inventory_agent, insight_agent)
    .add_edge(insight_agent, action_agent)     
    .build()
)
# Run the workflow with the user's initial message.
# For foundational clarity, use run (non streaming) and print the terminal event.
events = await workflow.run(prompt)

print('=== WORKFLOW OUTPUTS ===')
outputs = events.get_outputs()
# The outputs of the workflow are whatever the agents produce. So the outputs are expected to be a list
# of `AgentResponse` from the agents in the workflow.
outputs = cast(list[AgentResponse], outputs)
for output in outputs:
    print(f"{output.messages[0].author_name}: {output.text}\n")

# Summarize the final run state (e.g., COMPLETED)
print("Final state:", events.get_final_state())
print(events.get_outputs())




## Step 9 —  Run the workflow with your prompt - Simple Output

In [ ]:
from typing import cast
from agent_framework import WorkflowBuilder, Agent, AgentResponse
from agent_framework.azure import AzureAIAgentClient
from agent_framework_orchestrations import SequentialBuilder


prompt = (
    'TODO: replace with your initial prompt for the agent(s) in the workflow'
)

inventory_agent = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="InventoryAgent").as_agent(name="InventoryAgent")
insights_agent = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="InsightsAgent").as_agent(name="InsightsAgent")
actions_agent = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="ActionsAgent").as_agent(name="ActionsAgent")

# Run the workflow with the user's initial message.
workflow = SequentialBuilder(participants=[coagent, inventory_agent, insights_agent, actions_agent]).build()

# 3) Treat the workflow itself as an agent for follow-up invocations
agent = workflow.as_agent(name="my_wk_agent")
agent_response = await agent.run(prompt)
print('=== WORKFLOW OUTPUTS ===')
if agent_response.messages:
    print("\n===== Conversation =====")
    for i, msg in enumerate(agent_response.messages, start=1):
        name = msg.author_name or msg.role
        print(f"{'-' * 60}\n{i:02d} [{name}]\n{msg.text}")










# Optional - Run the workflow with your prompt - This is an example with with "Handoff" scenario. THIS is a HOME exercise 
## Step 10 — Orchestrate all agents with Microsoft Agent Framework (Python)

In [ ]:
from agent_framework import WorkflowBuilder
from agent_framework.azure import AzureAIAgentClient

# Handoff Orchestration Pattern: Dynamic Routing Based on Agent Output
# Each agent can decide if the workflow should continue or skip to a specific agent

# Scenario 1: Emergency Stockout Handoff
# If ProductInventoryManager detects critical stockout, bypass customer insights 
# and go directly to marketing for immediate action

# Scenario 2: Conditional Customer Analysis
# If SalesInsightAgent identifies high-value opportunity, trigger deeper 
# customer segmentation before marketing

# Scenario 3: Marketing Action Approval Loop
# If MarketingActionAgent proposes high-budget campaign, handoff back to 
# coordinator for human approval before execution

# Example: Build workflow with conditional handoff logic
# You can implement handoff by:
# 1) Using WorkflowBuilder with conditional edges
# 2) Creating a custom executor that examines agent output and routes accordingly
# 3) Using the SequentialBuilder with decision points

print("""
Handoff Orchestration Use Cases for Your 5-Agent Workflow:

1. **Critical Inventory Alert Handoff**
    - INventoryAegnt detects <10% stock on top SKU
    - Handoff: Skip CustomerInsight → Direct to MarketingAction
    - Action: Immediate promotion pause + supplier escalation

2. **High-Value Customer Opportunity Handoff**
    - CustomerInsightAgent identifies VIP segment affected by stockout
    - Handoff: Loop back to SalesInsightAgent for upsell alternatives
    - Action: Personalized offer generation before marketing


3. **Data Quality Handoff**
    - Any agent encounters incomplete data
    - Handoff: Escalate to another agent with specific data request
    - Action: Human-in-the-loop for data validation

4. **Multi-Channel Conflict Handoff**
    - MarketingActionAgent detects conflicting campaigns
    - Handoff: Back to SalesInsightAgent for priority ranking
    - Action: Re-sequence marketing actions by revenue impact

To implement, use WorkflowBuilder.add_conditional_edge() or
create custom executor logic that examines agent_response.text
for keywords like 'CRITICAL', 'APPROVAL_REQUIRED', 'DATA_MISSING'
""")

# Conceptual code structure (requires custom executor implementation):
# workflow = (
#     WorkflowBuilder(start_executor=coagent)
#     .add_edge(coagent, product_inventory_agent)
#     .add_conditional_edge(
#         product_inventory_agent,
#         lambda response: marketing_action_agent if "CRITICAL_STOCKOUT" in response.text 
#         else customer_insight_agent
#     )
#     .add_edge(customer_insight_agent, sales_agent)
#     .add_edge(sales_agent, marketing_action_agent)
#     .add_edge(marketing_action_agent, coagent, condition="APPROVAL_REQUIRED")
#     .build()
# )

## Step 21 — Cleanup (optional but recommended)
If you created SalesInsightsAgent only for the lab, delete it to avoid resource sprawl/cost.

In [ ]:
# Delete the agent you created in this lab (optional)
# project_client.agents.delete(agent_id=sales_agent_id)
# print('Deleted SalesInsightsAgent:', sales_agent_id)

## Validation Checklist
- ✅ Connected to Foundry project from Python
- ✅ Reused ProductInventoryManager by ID
- ✅ Created and tested SalesInsightsAgent
- ✅ Ran a sequential workflow orchestrating both agents